**Importy**

In [ ]:
!pip install transformers datasets kaggle numpy pandas nltk sentence-transformers faiss-cpu langchain langchain-groq

import os
import pandas as pd
import numpy as np
import nltk
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from sentence_transformers import SentenceTransformer
import faiss
import datasets
from collections import Counter
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.8 MB/s eta 0:00:00


**Pobranie "słów stopu" z NL toolkit**

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**Pobranie ze zbioru tekstu i zasobów leksykalnych "słów stopu" w j. ang.**

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

**Załadowanie zbioru danych i wyfiltrowanie pierwszych 200 rekordów o charakterze tekstowym. Wybrałem zbiór recenzji/opinii klientów Amazona na temat zamówionych produktów żywnościowych on-line.**

In [ ]:
df = pd.read_csv('/content/reddit.csv', usecols=["comment"], nrows=200)
import pandas as pd

**Podgląd danych**

In [ ]:
df.head()

,comment
0,"Whether you believe in climate change or not, ..."
1,"The problem isn't really the temperature, but ..."
2,OP this doesn't mean anything. No one is argui...
3,"Hey bud, 500 million years ago the Earth was u..."
4,"Yeah sure, Earths average temperatures has flu..."


**Czyścimy dataframe (z Hugging Face), wyrzucamy wszystko co nie jest tekstem (niepotrzebne w tym przypadku, ale potrzebne do innych zbiorów danych z większą ilością kolumn w róznych typach). Zmieniamy na listę w Pythonie, bierzemy 200 pierwszych rekordów i z połowy tworzymy zestaw treningowy. Reszta idzie do zbioru testowego.**

In [ ]:
df = df[['comment']].dropna().reset_index(drop=True)
docs = df['comment'].tolist()[:200]
train_docs = docs[:100]
test_docs = docs[100:200]

**Tworzymy funkcje na później. Funkcja, która esktraktuje "top 3" słowa klucze, które mają więcej niż 3 znaki (pomijając oczywiście słowa stopu) i wyrzuca je jako tokeny. Druga funkcja zmienia listę słów kluczowych w hashtagi.**

In [ ]:
def extract_keywords(text, num=3):
    tokens = re.findall(r"\b\w+\b", text.lower())
    tokens = [t for t in tokens if t not in stop_words and len(t) > 3]
    most_common = [word for word, _ in Counter(tokens).most_common(num)]
    return most_common

def keywords_to_hashtags(keywords):
    return [f"#{w}" for w in keywords]

**Tworzymy pipeline'a do podsumowań. Używamy standardowego modelu, np. t5-base lub bart-large. Tworzymy listę z podsumowaniami + podsumowania razem z hashtagami i słowami klucz. Potem tworzymy podsumowanie (summary), ekstraktujemy najlepsze słowa klucz, zmieniamy słowa klucz w hashtagi i dodajemy same podsumowania do końca listy "train_summaries", która posłuży do dotrenowania modelu. Na koniec zapisujemy podsumowanie razem z słowami klucz i hashtagami do późniejszego użytku.**

In [ ]:
summarizer = pipeline('summarization')

train_summaries = []
train_summaries_with_meta = []

for text in train_docs:
    output = summarizer(text, max_length=60, min_length=20, do_sample=False)
    summary_text = output[0]['summary_text']

    keywords = extract_keywords(summary_text, num=3)
    hashtags = keywords_to_hashtags(keywords)

    train_summaries.append(summary_text)

    train_summaries_with_meta.append({
        "summary": summary_text,
        "keywords": keywords,
        "hashtags": hashtags
    })

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Your max_length is set to 60, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Your max_length is set to 60, but your input_length is only 57. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Your max_length is set to 60, but your input_length is only 23. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)
Your max_length is set to 60, but your input_length is only 42. Since this is a summarization task, where outputs shorter 

**Bierzemy pierwsze 5 podsumowań razem z słowami klucz i hashtagami w celach weryfikacji**

In [ ]:
for i, meta in enumerate(train_summaries_with_meta[:5], start=1):
    print(f"Example {i} Summary: {meta['summary']}")
    print(f"  Keywords: {meta['keywords']}")
    print(f"  Hashtags: {meta['hashtags']}\n")

Example 1 Summary:  Climate change fear mongering is a little blown out of of perspective, but renewable energy is infinite . Solar panels are affordable and can lower your bill in half and sell energy back to the grind .
  Keywords: ['energy', 'climate', 'change']
  Hashtags: ['#energy', '#climate', '#change']

Example 2 Summary:  The problem isn't really the temperature, but the rate of temperature increase . The effect will be that ecosystems will have a hard time adapting fast enough, and many plant and animal species will go extinct . The biggest issues for people will be a change in what land can be framed and how
  Keywords: ['temperature', 'problem', 'really']
  Hashtags: ['#temperature', '#problem', '#really']

Example 3 Summary:  No one is arguing that humans are causing the planet to get too warm for the planet . Global warming is simply inconvenient for us. The earth goes through cycles of ice ages and glacial periods . By pumping greenhouse gases into the atmosphere we're 

**Finetuning modelu. Używamy t5-small (szybszy, ale też mniejszy) i ładujemy tokenizer oraz model z HuggingFace. Zmieniamy dane na HuggingFace type dataframe w celach treningowych. Zmieniamy tekst na tokeny (wartości numeryczne - tak aby model mógł analizować dane). Ładujemy zbiory, definiujemy hiperparametry i trenujemy model. Na koniec zapisujemy go jako model z dopiskiem "fine-tuned". W tym kroku użyłem również API z platformy o nazwie Weights & Biases. Udostępnia ona open-source klucze do użytku przy treniowaniu modelu.**

In [ ]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

train_dataset = datasets.Dataset.from_dict({
    'input_text': train_docs,
    'target_text': train_summaries
})

def preprocess_function(batch):
    inputs = tokenizer(batch["input_text"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(batch["target_text"], max_length=128, truncation=True, padding="max_length")
    batch["input_ids"] = inputs.input_ids
    batch["attention_mask"] = inputs.attention_mask
    batch["labels"] = targets.input_ids
    return batch

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="t5-fine-tuned",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

t5_output_dir = "t5-fine-tuned"
model.save_pretrained(t5_output_dir)
tokenizer.save_pretrained(t5_output_dir)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

/tmp/ipython-input-1952698790.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ultimaterollie (ultimaterollie-uniwersytet-swps) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,13.232100
20,10.104100
30,8.364400
40,7.115200
50,7.140100


('t5-fine-tuned/tokenizer_config.json',
 't5-fine-tuned/special_tokens_map.json',
 't5-fine-tuned/spiece.model',
 't5-fine-tuned/added_tokens.json',
 't5-fine-tuned/tokenizer.json')

**Ponownie ładujemy model, tym razem już właściwy (po fine-tuningu). Tworzymy listy z podsumowaniami i danymi (słowa klucz, hashtagi etc.) i generujemy podsumowanie z już wytrenowanego modelu. Ekstraktujemy nowe słowa kluczowe i hashtagi i zapisujemy do dalszego użytku.**

In [ ]:
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(t5_output_dir)
fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(t5_output_dir)
fine_tuned_summarizer = pipeline(
    'summarization',
    model=fine_tuned_model,
    tokenizer=fine_tuned_tokenizer
)

test_summaries = []
test_summaries_with_meta = []

for text in test_docs:
    output = fine_tuned_summarizer(text, max_length=60, min_length=20, do_sample=False)
    summary_text = output[0]['summary_text']

    keywords = extract_keywords(summary_text, num=3)
    hashtags = keywords_to_hashtags(keywords)

    test_summaries.append(summary_text)
    test_summaries_with_meta.append({
        "summary": summary_text,
        "keywords": keywords,
        "hashtags": hashtags
    })

Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Your max_length is set to 60, but your input_length is only 43. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=21)
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Your max_length is set to 60, but your input_length is only 42. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('..

**Ponownie wypisujemy 5 pierwszych słów klucz, hashtagi i podsumowania z naszego modelu.**

In [ ]:
for i, meta in enumerate(test_summaries_with_meta[:5], start=1):
    print(f"Test Example {i} Summary: {meta['summary']}")
    print(f"  Keywords: {meta['keywords']}")
    print(f"  Hashtags: {meta['hashtags']}\n")

Test Example 1 Summary: the 1800’s were called the little ice age . people think the earth is gonna explode or something . they don’t care about the climate, they care about your money .
  Keywords: ['care', '1800', 'called']
  Hashtags: ['#care', '#1800', '#called']

Test Example 2 Summary: as the planet heats up humans will probably be fucked . but the planet itself will be fine .
  Keywords: ['planet', 'heats', 'humans']
  Hashtags: ['#planet', '#heats', '#humans']

Test Example 3 Summary: if the earth starts to look like it did 500M years ago, we're all fucked as a species . scientists are trying to make the point .
  Keywords: ['earth', 'starts', 'look']
  Hashtags: ['#earth', '#starts', '#look']

Test Example 4 Summary: modular nuclear reactors and hydrogen are going ALL IN . Totally safe with no chance of melt down. clean power made where it is needed .
  Keywords: ['modular', 'nuclear', 'reactors']
  Hashtags: ['#modular', '#nuclear', '#reactors']

Test Example 5 Summary: clima

**Embeddujemy podsumowania z recenzji (tworzymy z nich wartości numeryczne potrzebne do naszego RAGa). Potem będziemy porównywać je na podstawie wektorów.**

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
summary_embeddings = embedder.encode(test_summaries, convert_to_numpy=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Używamy FAISS do stworzenia narzędzia umożliwiającego wyszukiwanie stworzonych wartości numerycznych. Pozwoli to na wyszukiwanie w podsumowaniach słów i ich znaczeń "najbliższych" do innych, aby móc zdefiniować ich przekaz.**

In [ ]:
dimension = summary_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(summary_embeddings))

**Stawiamy retirever. Pozwala on na podstawie inputu użytkownika znaleźć odpowiednie podsumowania z naszych danych.**

In [ ]:
def retrieve(query, k=5):
    query_emb = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_emb, k)
    return indices[0]

**Łączymy generator z retrieverem. Definiujemy funkcję, która pozwala nam znaleźć "topowe" podsumowania w zależności od kontekstu i oczywiście tego, o co pytamy model. Dajemy mu również kontekst, na którym ma bazować swoje odpowiedzi (proszę odpowiadaj we własnych słowach, używaj informacji kontekstowych itp. etc.). Na koniec funkcja zwraca nam odpowiedź modelu. Mamy postawionego RAGa :).**

In [ ]:
generator = pipeline(
    'text2text-generation',
    model=fine_tuned_model,
    tokenizer=fine_tuned_tokenizer
)


def generate_answer(query, k=5):
    topk_indices = retrieve(query, k)

    retrieved_contexts = [test_summaries[i] for i in topk_indices]

    context_block = "\n---\n".join(retrieved_contexts)
    prompt = (
        "Please answer the question below in your own words, using only the information provided in the context. "
        "Do not copy any sentences verbatim; summarize and paraphrase where needed.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}"
    )

    output = generator(prompt, max_length=200, do_sample=False)
    answer_text = output[0]['generated_text']
    return answer_text

Device set to use cuda:0


**Dałem mojemu modelowi 3 prompty. Odpowiedział poprawnie, jednak użył w każdym przypadku "żywych" przykładów. Można stwierdzić, że są one zgodne z tym o co pytamy i nie odbiegają od kontekstu. Model działa więc poprawnie, jednak użyję Groka do poprawnienia outputu.**

In [ ]:
queries = [
    "What problems do people mention?",
    "Are there mentions of global warming?",
    "Do the comments talk about temperature change?"
]

for q in queries:
    print("\nQuery:", q)
    ans = generate_answer(q, k=5)
    print("Answer:", ans)
    print("-"*40)

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: What problems do people mention?


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: the solutions from the left have almost nothing to do with it . the solutions won’t significantly affect it, and give more power to authority . --- the issue is freaking out about it and changing things over night . it's all knee jerk reactions and crisis mode . making laws to outlaw cars is nonsense without the infrastructure to support it . --- the main problem is that we put hundreds of trillions of dollars worth of effort into building things; countries, buildings, roads, farms; in places that are
----------------------------------------

Query: Are there mentions of global warming?


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: climate change is a natural planetary process . it's the effect that man has in that process that is being debated . --- climate change is real, but it's not the catastrophe it’s portrayed as . --- we agree that the rise in global temperatures is all man made . there is always an element of collectivism intertwined in the proposed solutions .
----------------------------------------

Query: Do the comments talk about temperature change?
Answer: average temperatures changing this rapidly will have negative effects . look at the slope of the line at the end . --- we agree that the rise in global temperatures is all man made . there is always an element of collectivism intertwined in the proposed solutions . --- climate change is a natural planetary process . there is always an element of collectivism intertwined in the proposed solutions .
----------------------------------------


**Próbujemy z Grokiem - działa :) Odpowiedzi są zgodne z podsumowaniami, które wcześniej czytaliśmym, a model odpowiada w sposób naturalny używając wnioskowania.**

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# 1. Wczytujemy plik .env
load_dotenv()

# 2. Pobieramy klucz Groq ze zmiennych środowiskowych
groq_api_key = os.getenv("GROQ_API_KEY")

# LangChain i Groq automatycznie szukają zmiennej środowiskowej o nazwie "GROQ_API_KEY",
# ale jeśli Twój kod wymaga jej przypisania do os.environ, robimy to tutaj:
os.environ["GROQ_API_KEY"] = groq_api_key

print("Klucz Groq załadowany pomyślnie!")

groq_model = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant")

def generate_answer(query, k=5):
    topk_indices = retrieve(query, k)

    retrieved_contexts = [test_summaries[i] for i in topk_indices]

    context_block = "\n---\n".join(retrieved_contexts)
    prompt = (
        "Please answer the question below based on the provided context. "
        "Synthesize the information into a concise and direct answer. "
        "Do not simply repeat sentences from the context.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}"
    )

    response = groq_model.invoke(prompt)
    return response.content

queries = [
    "What percent of the comments mention fear? Could you count the comments?",
    "Are there mentions of global warming?",
    "Do the comments talk about temperature change?"
]

for q in queries:
    print("\nQuery:", q)
    ans = generate_answer(q, k=5)
    print("Answer:", ans)
    print("-"*40)


Query: What percent of the comments mention fear? Could you count the comments?
Answer: There are 5 comments in the context provided. Mention of "fear" is present in the following statement: "the toxic combination of fear mongering..."
----------------------------------------

Query: Are there mentions of global warming?
Answer: Yes, there is a mention of "global warming" in the context, specifically that it was referred to as "global warming" until around 5-6 years ago, before being referred to as "climate change".
----------------------------------------

Query: Do the comments talk about temperature change?
Answer: Yes, the comments discuss temperature change, specifically the rate and causes of global temperature changes.
----------------------------------------
